# MXFrame: DataFrames powered by Mojo

> A minimal tour of MXFrame's Python and SQL frontends, followed by a pandas comparison at scale.



MXFrame is a lazy DataFrame query engine with a familiar Python API, SQL support, Apache Arrow data, and precompiled Mojo kernels for CPU and GPU execution. It exists to make compiled analytics kernels accessible without making users write low-level GPU code.



In this notebook you will:

1. install MXFrame in Google Colab;

2. create data and run one Python API query;

3. run a realistic SQL join; and

4. benchmark pandas and MXFrame on the same large dataset.



The current GPU build is validated on NVIDIA hardware. Mojo's portable kernel design can target other accelerators, but MXFrame's AMD and Apple Silicon validation is still on the roadmap.



For internals, query planning, and the full benchmark methodology, use the [MXFrame README](https://github.com/abhisheksreesaila/mxframe) and the [interactive pipeline visualizer](../../visualize/mxframe_pipeline.html).

## 1. Install



This tutorial environment pins **MXFrame 0.5.0**. In Google Colab, choose a GPU runtime, install the runtime and SQL extras, then restart the runtime once.



MXFrame also runs on CPU. Its CPU path benefits from precompiled Mojo AOT kernels, so GPU acceleration is an option rather than a requirement.



> Local VS Code users should select the **Python (mojo-gpu-tutorials)** kernel; the project Pixi environment already contains the required packages.

In [1]:
# Google Colab only: uncomment, run, then restart the runtime.

# %pip install "mxframe[runtime,sql]==0.5.0" -q



import importlib.metadata as metadata

import sys



print(f"MXFrame {metadata.version('mxframe')} | Python {sys.version.split()[0]}")

MXFrame 0.5.0 | Python 3.12.13


## 2. Why MXFrame?



Pandas is an excellent general-purpose DataFrame library, but its execution is CPU-oriented. MXFrame keeps a Python-friendly, lazy query API while dispatching analytical work to precompiled **Mojo AOT kernels** on CPU or GPU. Apache Arrow provides typed columnar storage, and SQL is compiled into the same logical query plan as the Python API.



That combination matters most for large analytical workloads: filters, aggregations, sorts, and especially joins over millions of rows. Small workloads may remain faster on CPU because transferring data to a GPU has a fixed cost.

## 3. Python frontend



Create an Arrow table, describe a lazy query, and call `.compute()` once. The query filters sales, groups them by region, and calculates revenue statistics.

In [2]:
import pyarrow as pa

from mxframe import LazyFrame, Scan, col, lit



def pick_device() -> str:

    try:

        from max import driver

        return "gpu" if driver.accelerator_count() > 0 else "cpu"

    except Exception:

        return "cpu"



DEVICE = pick_device()

print(f"Using: {DEVICE.upper()}")



sales = pa.table({

    "region": pa.array(["North", "South", "North", "West", "South", "West"]),

    "revenue": pa.array([120.0, 90.0, 180.0, 75.0, 140.0, 110.0], pa.float32()),

    "units": pa.array([4, 3, 6, 2, 5, 4], pa.int32()),

})



sales_query = (

    LazyFrame(Scan(sales))

    .filter(col("revenue") >= lit(100.0))

    .groupby("region")

    .agg(

        col("revenue").sum().alias("total_revenue"),

        col("units").sum().alias("units_sold"),

    )

    .sort(col("total_revenue"), descending=True)

)



sales_result = sales_query.compute(device=DEVICE)

sales_result.to_pandas()

Using: GPU


,region,total_revenue,units_sold
0,North,300.0,10.0
1,South,140.0,5.0
2,West,110.0,4.0


## 4. SQL frontend: a real join



MXFrame accepts Arrow tables as named SQL inputs. This query joins orders to products, computes line revenue, groups by category, and sorts the result. It is still lazy until `.compute()`.

In [3]:
from mxframe import sql



orders = pa.table({

    "order_id": pa.array([1, 2, 3, 4, 5, 6], pa.int32()),

    "product_id": pa.array([10, 20, 10, 30, 20, 30], pa.int32()),

    "quantity": pa.array([2, 1, 5, 3, 2, 4], pa.int32()),

    "unit_price": pa.array([15.0, 40.0, 15.0, 25.0, 40.0, 25.0], pa.float32()),

})



products = pa.table({

    "product_id": pa.array([10, 20, 30], pa.int32()),

    "category": pa.array(["Hardware", "Electronics", "Hardware"]),

})



revenue_by_category = sql(

    """

    SELECT

        p.category,

        SUM(o.quantity * o.unit_price) AS total_revenue,

        COUNT(*) AS order_lines

    FROM orders AS o

    JOIN products AS p ON o.product_id = p.product_id

    GROUP BY p.category

    ORDER BY total_revenue DESC

    """,

    orders=orders,

    products=products,

).compute(device=DEVICE)



revenue_by_category.to_pandas()

,category,total_revenue,order_lines
0,Hardware,280.0,4.0
1,Electronics,120.0,2.0


## 5. pandas vs MXFrame at 10 million rows



GPU acceleration pays off when enough parallel work amortizes data-transfer and dispatch costs. The MXFrame benchmark documentation highlights join-heavy analytical queries at 10 million rows, so this benchmark uses a large integer-key join followed by grouped revenue aggregation.



The comparison uses the same Arrow source data, excludes Arrow-to-pandas conversion from timing, performs one warm-up, and reports the median of three runs. Results depend on your CPU, GPU, memory bandwidth, and MXFrame version; run the cell rather than treating any saved number as universal.



> The published MXFrame suite also shows that the **CPU path can win**: its precompiled Mojo kernels can outperform conventional DataFrame execution even when GPU dispatch is not worthwhile.

### Build the benchmark data



Ten million fact rows are large enough to expose parallelism while remaining practical on a typical Colab GPU runtime. Reduce `ROWS` if your runtime has limited memory.

In [4]:
import numpy as np



ROWS = 10_000_000

PRODUCTS = 4_096

CATEGORIES = 16

rng = np.random.default_rng(42)



orders_large = pa.table({

    "product_id": pa.array(rng.integers(0, PRODUCTS, ROWS, dtype=np.int32)),

    "quantity": pa.array(rng.integers(1, 8, ROWS).astype(np.float32)),

})



product_ids = np.arange(PRODUCTS, dtype=np.int32)

products_large = pa.table({

    "product_id": pa.array(product_ids),

    "category_id": pa.array(product_ids % CATEGORIES),

    "unit_price": pa.array(rng.uniform(5.0, 250.0, PRODUCTS).astype(np.float32)),

})



# Prepare pandas inputs before timing so both engines start from ready columnar data.

orders_pd = orders_large.to_pandas()

products_pd = products_large.to_pandas()



print(f"{ROWS:,} orders | {PRODUCTS:,} products | {CATEGORIES} categories")

10,000,000 orders | 4,096 products | 16 categories


### Run the same join and aggregation



Each engine joins orders to product metadata, calculates `quantity * unit_price`, then sums revenue and counts lines by category. Lower time is better.

In [5]:
import gc

import statistics

import time

from IPython.display import HTML, display



def pandas_query():

    joined = orders_pd.merge(products_pd, on="product_id", how="inner", sort=False)

    joined["revenue"] = joined["quantity"] * joined["unit_price"]

    return joined.groupby("category_id", sort=False).agg(

        total_revenue=("revenue", "sum"),

        order_lines=("product_id", "count"),

    )



def mxframe_query(device: str):

    return (

        LazyFrame(Scan(orders_large))

        .join(

            LazyFrame(Scan(products_large)),

            left_on="product_id",

            right_on="product_id",

            how="inner",

        )

        .with_columns((col("quantity") * col("unit_price")).alias("revenue"))

        .groupby("category_id")

        .agg(

            col("revenue").sum().alias("total_revenue"),

            col("product_id").count().alias("order_lines"),

        )

        .compute(device=device)

    )



def median_seconds(function, runs=3):

    times = []

    for _ in range(runs):

        gc.collect()

        start = time.perf_counter()

        function()

        times.append(time.perf_counter() - start)

    return statistics.median(times)



# Warm each path once; timed runs measure steady-state analytical execution.

pandas_query()

mxframe_query("cpu")

if DEVICE == "gpu":

    mxframe_query("gpu")



timings = {

    "pandas": median_seconds(pandas_query),

    "MXFrame CPU": median_seconds(lambda: mxframe_query("cpu")),

}

if DEVICE == "gpu":

    timings["MXFrame GPU"] = median_seconds(lambda: mxframe_query("gpu"))



best_mx_name = min((name for name in timings if name.startswith("MXFrame")), key=timings.get)

speedup = timings["pandas"] / timings[best_mx_name]

max_time = max(timings.values())

colors = {"pandas": "#64748b", "MXFrame CPU": "#0f766e", "MXFrame GPU": "#e8590c"}

bars = "".join(

    f"<div style='margin:12px 0'>"

    f"<div style='display:flex;justify-content:space-between'><strong>{name}</strong><span>{seconds:.3f} s</span></div>"

    f"<div style='height:18px;background:#e5e7eb;border-radius:3px'>"

    f"<div style='height:18px;width:{100 * seconds / max_time:.1f}%;background:{colors[name]};border-radius:3px'></div>"

    f"</div></div>"

    for name, seconds in timings.items()

)

display(HTML(

    "<div style='max-width:720px;border:1px solid #d1d5db;padding:20px;font-family:sans-serif'>"

    "<h3 style='margin:0 0 4px'>10M-row join + aggregation</h3>"

    "<p style='margin:0 0 16px;color:#475569'>Warm median of 3 runs; lower is better</p>"

    f"{bars}"

    f"<p style='margin:18px 0 0;font-size:18px'><strong>{best_mx_name}: {speedup:.2f}x vs pandas</strong></p>"

    "</div>"

))

## Reading the result



A GPU win here means this large, parallel join-and-aggregation workload amortized dispatch and transfer costs on your hardware. It does **not** mean every DataFrame query belongs on a GPU: small inputs and low-work operations often favor CPU execution.



MXFrame's CPU backend is also part of the story. It uses precompiled Mojo kernels and can be the fastest MXFrame path when GPU overhead is larger than the work itself.



Continue with the [MXFrame README and benchmark methodology](https://github.com/abhisheksreesaila/mxframe) or open the [interactive query pipeline](../../visualize/mxframe_pipeline.html) for the architecture walkthrough.